# Clase 01 — Del notebook a la producción

**Curso:** MLOps y Monitorización con Python · **Bloque 1:** El problema y la base
**Tiempo estimado:** 90 minutos · **Capítulo del manual:** `manual/01_del_notebook_a_produccion.md`

## Qué vas a ver en esta clase

- El **ciclo de vida** de un sistema de ML y por qué "el modelo" es solo una pieza pequeña.
- La **deuda técnica específica del ML** (Sculley et al., 2015): acoplamiento de configuraciones, dependencias de datos ocultas, *entanglement*, *training-serving skew*.
- Los **niveles de madurez MLOps** de Google Cloud y de Microsoft Azure, comparados.
- Una **inspección guiada, con código**, del notebook `datos/notebook_sucio.ipynb`: lo abrimos como datos (sin ejecutarlo) y contamos sus problemas.
- Una **demostración en vivo**: el mismo cálculo da un resultado distinto según qué celda se ejecutó en último lugar.
- La **exactitud engañosa**: con clases desbalanceadas, un modelo que no aprende nada puede parecer que "funciona bien".

Este notebook **no entrena ningún modelo**: es deliberadamente ligero. El notebook sucio que analizamos es material de diagnóstico, no algo que debamos ejecutar de principio a fin.

## Requisitos

- Ninguna librería nueva: `numpy` y la librería estándar (`json`, `os`, `re`). La celda siguiente instala solo lo que falte (funciona igual en Google Colab).
- La carpeta `datos/` del curso, con `notebook_sucio.ipynb` dentro. En Colab, súbela desde el panel de archivos.

> **Cómo usar este notebook:** ejecuta las celdas en orden, de arriba a abajo, una sola vez.

---
## Instalación

Instala solo lo que falte. En un entorno con todo instalado no hace nada.

In [ ]:
import subprocess, sys


def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
print("Librerías listas.")

---
## Celda 0 — Setup

Definimos `cargar_datos` y abrimos `notebook_sucio.ipynb` **como datos**: lo leemos con `json.load`, igual que harías con cualquier archivo JSON. No lo ejecutamos. Más adelante, en la demostración en vivo, usaremos un generador de números aleatorios con semilla propia (`np.random.default_rng(42)`), así que no hace falta fijar aquí ninguna semilla global.

In [ ]:
import os
import json
import re

import numpy as np

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

RUTA_SUCIO = cargar_datos("notebook_sucio.ipynb")
sucio = json.load(open(RUTA_SUCIO, encoding="utf-8"))
print("Notebook cargado:", RUTA_SUCIO)
print("Celdas totales:", len(sucio["cells"]))

---
## 1. El ciclo de vida y "el modelo no es el sistema"

Antes de mirar el notebook sucio, recuerda el mapa del capítulo: **idea → datos → entrenamiento → validación → despliegue → monitoreo → mantenimiento**, en bucle. El notebook que vamos a inspeccionar solo intenta cubrir "datos" y "entrenamiento", una vez, sin bucle y sin las otras cinco fases. Esa es, en sí misma, la primera observación del análisis.

---
## 2. Inspección de la estructura, sin ejecutar nada

Contamos celdas de código y de markdown, y buscamos con una expresión regular los indicios más fáciles de detectar mecánicamente: rutas absolutas de un sistema operativo concreto.

In [ ]:
tipos = [celda["cell_type"] for celda in sucio["cells"]]
n_code = tipos.count("code")
n_md = tipos.count("markdown")
print("código:", n_code, "| markdown:", n_md)

PATRON_RUTA = re.compile(r"(/Users/\w+|/home/\w+)[\w/.]*")
rutas = []
for i, celda in enumerate(sucio["cells"]):
    if celda["cell_type"] != "code":
        continue
    fuente = "".join(celda["source"])
    for m in PATRON_RUTA.finditer(fuente):
        rutas.append((i + 1, m.group(0)))

print("rutas absolutas encontradas:", len(rutas))
for celda_n, ruta in rutas:
    print("  celda", celda_n, "→", ruta)

**¿Qué observas?** El notebook tiene 16 celdas (11 de código, 5 de markdown) y contiene **2 rutas absolutas hardcodeadas**, ambas del portátil de una persona concreta: una para cargar un archivo "extra" (celda 3) y otra para guardar el modelo (celda 16). Ninguna de las dos funcionaría en tu máquina tal cual está.

---
## 3. Demostración en vivo: el orden de ejecución cambia el resultado

En el notebook sucio, la celda 6 fija `umbral_riesgo = 0.5`, la celda 8 lo sobrescribe a `0.8` "para probar", y la celda 9 usa `umbral_riesgo` para contar clientes en riesgo. El resultado de la celda 9 depende de **cuál de las dos celdas (6 u 8) se ejecutó en último lugar**, no del orden en que aparecen en la pantalla.

Reproducimos aquí el mismo mecanismo con datos mínimos y una semilla fija, para verlo con números concretos y reproducibles (a diferencia del notebook sucio, aquí sí controlamos la aleatoriedad).

In [ ]:
rng = np.random.default_rng(42)
prob_estimada_demo = rng.random(20)          # 20 "clientes" de juguete, probabilidad de abandono simulada


def contar_en_riesgo(umbral):
    """Mismo código que la celda 9 del notebook sucio, aislado en una función."""
    return int((prob_estimada_demo > umbral).sum())


# Orden A: la última celda "de umbral" ejecutada fijó 0.5
umbral_riesgo = 0.5
resultado_orden_a = contar_en_riesgo(umbral_riesgo)

# Orden B: alguien vuelve a ejecutar la celda de prueba con 0.8, sin tocar el resto
umbral_riesgo = 0.8
resultado_orden_b = contar_en_riesgo(umbral_riesgo)

print("mismo notebook, mismo código de conteo:")
print("  umbral_riesgo = 0.5  →", resultado_orden_a, "clientes en riesgo")
print("  umbral_riesgo = 0.8  →", resultado_orden_b, "clientes en riesgo")
assert resultado_orden_a != resultado_orden_b

**¿Qué observas?** Con los mismos 20 clientes de juguete, `contar_en_riesgo` da 12 clientes en riesgo si la última celda de umbral ejecutada fue la de `0.5`, y 5 si fue la de `0.8`. El código de la celda 9 (aquí, `contar_en_riesgo`) no cambió en ningún momento: lo único que cambió fue **qué celda se ejecutó en último lugar**. Esto es exactamente el síntoma de *entanglement* de la sección 4 del capítulo: cambiar "solo una celda" altera un resultado que aparece varias celdas después, sin ningún aviso.

---
## 4. La exactitud engañosa: otro problema que no se ve a simple vista

La celda 14 del notebook sucio imprime `modelo.score(...)` (exactitud) como si esa cifra bastara para decir "el modelo funciona". No lo hace: si el abandono es minoritario, un modelo que casi nunca predice "abandona" puede tener una exactitud altísima sin haber aprendido nada útil. Lo comprobamos con un ejemplo mínimo, sin entrenar nada.

In [ ]:
# 100 clientes de juguete, 12 de los cuales abandonan (ejemplo ilustrativo, sin entrenar nada)
abandono_ejemplo = np.array([0] * 88 + [1] * 12)
prediccion_trivial = np.zeros_like(abandono_ejemplo)          # "nunca predice abandono"

exactitud_trivial = (prediccion_trivial == abandono_ejemplo).mean()
abandonos_reales = int(abandono_ejemplo.sum())
abandonos_detectados = int(((prediccion_trivial == 1) & (abandono_ejemplo == 1)).sum())

print("tasa de abandono:", abandono_ejemplo.mean())
print("exactitud de un modelo que nunca predice abandono:", exactitud_trivial)
print(f"de los {abandonos_reales} abandonos reales, ese modelo detecta:", abandonos_detectados)
assert exactitud_trivial > 0.85 and abandonos_detectados == 0

**¿Qué observas?** Un modelo que **nunca predice abandono** acierta el 88 % de las veces (porque el 88 % de los clientes, en efecto, no abandona) y detecta 0 de los 12 abandonos reales. Si la celda 14 del notebook sucio solo mira `modelo.score(...)`, un resultado así de inútil puede pasar por "el modelo funciona bien". Hace falta, como mínimo, mirar cuántos abandonos reales detecta el modelo (esto se retoma con métricas propias en la clase 6).

---
## 5. ¿En qué nivel de madurez está este proyecto?

Con lo que acabamos de observar, y con el resto de problemas descritos en el capítulo (sin semilla en la generación de datos ni en el `train_test_split`, una celda que solo funciona en un orden distinto del visual, una dependencia de datos oculta, un modelo entrenado y descartado, cero pruebas automáticas), el notebook sucio es un ejemplo de **Nivel 0** tanto en la escala de Google Cloud (todo manual, sin *pipeline*) como en la de Microsoft Azure (Nivel 0 — Sin MLOps). La clase 2 retoma este mismo archivo y empieza a subirlo de nivel.

---
## Resumen de la clase

- Un **sistema de ML** es mucho más que "el modelo": datos, configuración, servicio, monitoreo y mantenimiento, en un ciclo que se repite.
- La **deuda técnica del ML** tiene formas propias: acoplamiento de configuraciones, dependencias de datos ocultas, *entanglement*, *training-serving skew*.
- Los **niveles de madurez MLOps** (Google Cloud: 3 niveles; Azure: 5 niveles) sitúan un proyecto según cuánto de su ciclo de vida está automatizado.
- El notebook `datos/notebook_sucio.ipynb` es Nivel 0 de libro: ruta absoluta hardcodeada, sin semillas, una celda que depende de un orden de ejecución distinto del visual, una variable global sobrescrita, una dependencia de datos oculta, sin validación de datos, una exactitud que no dice nada sobre si el modelo aprendió algo, un modelo que se entrena y se tira, y cero pruebas.

*Continúa con la clase 2: Reproducibilidad.*